testデータセットを使ったテスト

In [7]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models  # ★ ConvNeXtを呼び出すために追加
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# --- 1. 設定: ファイルパスとパラメータ ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/1201_humomentstest'

TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_upsamp/train_max_upsampledx.csv'
TEST_CSV_PATH  = f'{pre}/src/FF/AFF/dataset_upsamp/test_fixed.csv'

TARGET_EPOCH = 100
SAVE_DIR = f"../save/CBAMtest/ConvNext_{TARGET_EPOCH}epoch_fixed_dataset"
MODEL_WEIGHT_PATH = f'{SAVE_DIR}/best_fusion_model.pth'


# --- 2. モデルのクラス定義 (ConvNeXt仕様に修正) ---
class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super(CBAM, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels)
        )
        self.sigmoid_channel = nn.Sigmoid()
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid_spatial = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(torch.mean(x, dim=(2, 3)))
        max_out = self.fc(torch.amax(x, dim=(2, 3)))
        channel_weight = self.sigmoid_channel(avg_out + max_out).unsqueeze(-1).unsqueeze(-1)
        x_c = x * channel_weight
        
        avg_out_s = torch.mean(x_c, dim=1, keepdim=True)
        max_out_s = torch.amax(x_c, dim=1, keepdim=True)
        spatial_weight = self.sigmoid_spatial(self.conv_spatial(torch.cat([avg_out_s, max_out_s], dim=1)))
        
        out_feat = x_c * spatial_weight
        attention_weight = channel_weight * spatial_weight
        return out_feat, attention_weight

# ※ ResBlockクラスは使用しないため削除しました

class FusionModel(nn.Module):
    def __init__(self, num_classes, fusion_strategy='AFF'):
        super(FusionModel, self).__init__()
        self.fusion_strategy = fusion_strategy
        
        # ★ CNN部分をConvNeXt-Tinyの特徴抽出器（features）に差し替え
        convnext = models.convnext_tiny(weights=None)
        self.dl_extractor = convnext.features 
        
        # ★ ConvNeXt-Tinyの最終出力チャネル数は「768」なので、各層の次元を768に合わせる
        self.hc_projector = nn.Sequential(nn.Linear(10, 768), nn.ReLU(inplace=True))
        self.fusion_attention = CBAM(768)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(768, num_classes)
        )

    def forward(self, img, hc_vec, return_attention=False):
        Y = self.dl_extractor(img)
        B, C, H, W = Y.shape
        X = self.hc_projector(hc_vec).view(B, C, 1, 1).expand(B, C, H, W)
        
        out_feat, attention_weight = self.fusion_attention(X + Y)
        
        fused = out_feat * X + (1 - out_feat) * Y
        out = self.classifier(fused)
        
        if return_attention:
            return out, attention_weight
        return out


# --- 3. Datasetの定義 ---
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler=None, transform=None, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        
        features = self.df[self.feature_cols].values
        
        if is_train:
            self.scaler = StandardScaler()
            self.hc_features = self.scaler.fit_transform(features).astype('float32')
        else:
            assert scaler is not None, "Test set requires a fitted scaler from training set."
            self.scaler = scaler
            self.hc_features = self.scaler.transform(features).astype('float32')
            
        self.labels = self.df['Label'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_root, 'mask', self.df.iloc[idx]['category'], self.df.iloc[idx]['filename'])
        
        try:
            image = Image.open(img_path).convert('RGB')
        except (OSError, FileNotFoundError):
            print(f"Error loading: {img_path}")
            image = Image.new('RGB', (256, 256), (0, 0, 0))
            
        if self.transform:
            image = self.transform(image)
            
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, hc_feat, label


# --- 4. 可視化関数 ---
def save_attention_heatmaps(model, dataloader, device, save_dir, num_samples=10):
    model.eval()
    heatmap_dir = os.path.join(save_dir, 'heatmaps')
    os.makedirs(heatmap_dir, exist_ok=True)
    
    inv_normalize = transforms.Normalize(
        mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
        std=[1/0.229, 1/0.224, 1/0.225]
    )

    saved_count = 0
    print(f"\nGenerating and saving {num_samples} heatmaps to {heatmap_dir}...")

    with torch.no_grad():
        for images, hc_feats, labels in dataloader:
            images = images.to(device)
            hc_feats = hc_feats.to(device)
            
            outputs, attention_weights = model(images, hc_feats, return_attention=True)
            _, preds = torch.max(outputs, 1)

            for i in range(images.size(0)):
                if saved_count >= num_samples:
                    return

                # 元の画像サイズ（例: 256x256）を取得
                orig_h, orig_w = images[i].shape[1], images[i].shape[2]

                img_display = inv_normalize(images[i].cpu())
                img_display = img_display.permute(1, 2, 0).numpy()
                img_display = np.clip(img_display, 0, 1)

                # アテンションマップを元の画像サイズに拡大（補間）する
                att_tensor = attention_weights[i].mean(dim=0, keepdim=True).unsqueeze(0)
                att_tensor_resized = F.interpolate(att_tensor, size=(orig_h, orig_w), mode='bilinear', align_corners=False)
                att_map = att_tensor_resized.squeeze().cpu().numpy()

                plt.figure(figsize=(15, 5))
                
                # 1. 元画像
                plt.subplot(1, 3, 1)
                plt.imshow(img_display)
                plt.title(f"True: {labels[i].item()} | Pred: {preds[i].item()}")
                plt.axis('off')
                
                # 2. ヒートマップ単体
                plt.subplot(1, 3, 2)
                im = plt.imshow(att_map, cmap='binary', vmin=0, vmax=1)
                plt.title("Fusion Attention Ratio (W->B)")
                cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
                cbar.set_label('Black: Hand-Crafted / White: CNN', rotation=270, labelpad=15)
                plt.axis('off')
                
                # 3. 重ね合わせ
                plt.subplot(1, 3, 3)
                plt.imshow(img_display)
                plt.imshow(att_map, cmap='binary', alpha=0.5, vmin=0, vmax=1)
                plt.title("Overlay")
                plt.axis('off')

                plt.tight_layout()
                plt.savefig(os.path.join(heatmap_dir, f'heatmap_sample_{saved_count+1}.png'))
                plt.close()
                saved_count += 1


# --- 5. メインテスト関数 ---
def evaluate_model():
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    if not os.path.exists(MODEL_WEIGHT_PATH):
        print(f"Error: Model weight not found at {MODEL_WEIGHT_PATH}")
        return

    if not os.path.exists(TEST_CSV_PATH):
        print(f"Error: Test CSV not found at {TEST_CSV_PATH}")
        return
    
    if not os.path.exists(TRAIN_CSV_PATH):
        print(f"Error: Train CSV not found at {TRAIN_CSV_PATH}. It is needed to fit the StandardScaler.")
        return

    print("Loading Datasets...")
    
    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_df = pd.read_csv(TRAIN_CSV_PATH)
    train_dataset = CustomMultiModalDataset(train_df, IMG_ROOT, transform=transform, is_train=True)
    
    test_df = pd.read_csv(TEST_CSV_PATH)
    test_dataset = CustomMultiModalDataset(test_df, IMG_ROOT, scaler=train_dataset.scaler, transform=transform, is_train=False)
    test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)
    
    print(f"Test Data: {len(test_df)} samples")
    print("-" * 30)

    # ★ モデルを初期化し、重みをロード
    model = FusionModel(num_classes=3, fusion_strategy='AFF').to(DEVICE)
    model.load_state_dict(torch.load(MODEL_WEIGHT_PATH, map_location=DEVICE))
    model.eval()

    all_preds = []
    all_labels = []

    print(f"Starting Evaluation on {DEVICE}...")

    with torch.no_grad():
        for images, hc_feats, labels in test_loader:
            images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(images, hc_feats)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    print(f"\n=== Test Results ===")
    print(f"Overall Accuracy: {accuracy:.4f}\n")
    
    print("Classification Report:")
    print(classification_report(all_labels, all_preds, digits=4))

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix (Test Data)')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    
    os.makedirs(SAVE_DIR, exist_ok=True)
    cm_save_path = f'{SAVE_DIR}/test_confusion_matrix.png'
    plt.savefig(cm_save_path)
    plt.close()
    print(f"Confusion Matrix saved to {cm_save_path}")

    save_attention_heatmaps(model, test_loader, DEVICE, SAVE_DIR, num_samples=10)
    print("All tasks completed.")

# --- 実行 ---
if __name__ == "__main__":
    evaluate_model()

Error: Train CSV not found at /home/hiromu/energy/src/FF/AFF/dataset_upsamp/train_max_upsampledx.csv. It is needed to fit the StandardScaler.


7×7で出力

In [13]:
import os
import torch
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# --- 1. 設定: ファイルパスとパラメータ ---
pre = "/home/hiromu/energy"
# IMG_ROOT = f'{pre}/data/1201_humomentstest'
IMG_ROOT = f'{pre}/data/26_0413_clean'

TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_upsamp/train_max_upsampled.csv'
TEST_CSV_PATH  = f'{pre}/src/FF/AFF/dataset_upsamp/test_fixed.csv'
# TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_upsamp_clean/train_max_upsampled.csv'
# TEST_CSV_PATH  = f'{pre}/src/FF/AFF/dataset_upsamp_clean/test_fixed.csv'

TARGET_EPOCH = 100
SAVE_DIR = f"../save/CBAMtest/ConvNext_{TARGET_EPOCH}epoch_fixed_dataset"
# SAVE_DIR = f"../save/nomal/ConvNext_{TARGET_EPOCH}epoch_fixed_dataset"

MODEL_WEIGHT_PATH = f'{SAVE_DIR}/best_fusion_model.pth'


# --- 2. モデルのクラス定義 (ConvNeXt仕様に修正) ---
class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super(CBAM, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels)
        )
        self.sigmoid_channel = nn.Sigmoid()
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid_spatial = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(torch.mean(x, dim=(2, 3)))
        max_out = self.fc(torch.amax(x, dim=(2, 3)))
        channel_weight = self.sigmoid_channel(avg_out + max_out).unsqueeze(-1).unsqueeze(-1)
        x_c = x * channel_weight
        
        avg_out_s = torch.mean(x_c, dim=1, keepdim=True)
        max_out_s = torch.amax(x_c, dim=1, keepdim=True)
        spatial_weight = self.sigmoid_spatial(self.conv_spatial(torch.cat([avg_out_s, max_out_s], dim=1)))
        
        out_feat = x_c * spatial_weight
        attention_weight = channel_weight * spatial_weight
        return out_feat, attention_weight


class FusionModel(nn.Module):
    def __init__(self, num_classes, fusion_strategy='AFF'):
        super(FusionModel, self).__init__()
        self.fusion_strategy = fusion_strategy
        
        convnext = models.convnext_tiny(weights=None)
        self.dl_extractor = convnext.features 
        
        self.hc_projector = nn.Sequential(nn.Linear(10, 768), nn.ReLU(inplace=True))
        self.fusion_attention = CBAM(768)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(768, num_classes)
        )

    def forward(self, img, hc_vec, return_attention=False):
        Y = self.dl_extractor(img)
        B, C, H, W = Y.shape
        X = self.hc_projector(hc_vec).view(B, C, 1, 1).expand(B, C, H, W)
        
        out_feat, attention_weight = self.fusion_attention(X + Y)
        
        fused = out_feat * X + (1 - out_feat) * Y
        out = self.classifier(fused)
        
        if return_attention:
            return out, attention_weight
        return out


# --- 3. Datasetの定義 ---
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler=None, transform=None, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        
        features = self.df[self.feature_cols].values
        
        if is_train:
            self.scaler = StandardScaler()
            self.hc_features = self.scaler.fit_transform(features).astype('float32')
        else:
            assert scaler is not None, "Test set requires a fitted scaler from training set."
            self.scaler = scaler
            self.hc_features = self.scaler.transform(features).astype('float32')
            
        self.labels = self.df['Label'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_root, 'mask', self.df.iloc[idx]['category'], self.df.iloc[idx]['filename'])
        
        try:
            image = Image.open(img_path).convert('RGB')
        except (OSError, FileNotFoundError):
            print(f"Error loading: {img_path}")
            image = Image.new('RGB', (256, 256), (0, 0, 0))
            
        if self.transform:
            image = self.transform(image)
            
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, hc_feat, label


# --- 4. 可視化関数 (生のアテンションマップのみ出力) ---
def save_attention_heatmaps(model, dataloader, device, save_dir, num_samples=10):
    model.eval()
    heatmap_dir = os.path.join(save_dir, 'heatmaps_raw')
    os.makedirs(heatmap_dir, exist_ok=True)

    saved_count = 0
    print(f"\nGenerating and saving {num_samples} raw 7x7 heatmaps to {heatmap_dir}...")

    with torch.no_grad():
        for images, hc_feats, labels in dataloader:
            images = images.to(device)
            hc_feats = hc_feats.to(device)
            
            outputs, attention_weights = model(images, hc_feats, return_attention=True)
            _, preds = torch.max(outputs, 1)

            for i in range(images.size(0)):
                if saved_count >= num_samples:
                    return

                # 補間を行わず、モデル出力そのままの次元(例: 7x7)で取得
                att_map = attention_weights[i].mean(dim=0).cpu().numpy()

                plt.figure(figsize=(6, 5))
                
                # 7x7のヒートマップ単体をそのまま描画
                im = plt.imshow(att_map, cmap='binary', vmin=0, vmax=1)
                plt.title(f"Raw Attention Ratio\nTrue: {labels[i].item()} | Pred: {preds[i].item()}")
                
                cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
                cbar.set_label('Black: Hand-Crafted / White: CNN', rotation=270, labelpad=15)
                
                # グリッドのピクセルがわかりやすいように軸目盛りを消す
                plt.axis('off')

                plt.tight_layout()
                plt.savefig(os.path.join(heatmap_dir, f'heatmap_raw_sample_{saved_count+1}.png'))
                plt.close()
                saved_count += 1


# --- 5. メインテスト関数 ---
def evaluate_model():
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    if not os.path.exists(MODEL_WEIGHT_PATH):
        print(f"Error: Model weight not found at {MODEL_WEIGHT_PATH}")
        return

    if not os.path.exists(TEST_CSV_PATH):
        print(f"Error: Test CSV not found at {TEST_CSV_PATH}")
        return
    
    if not os.path.exists(TRAIN_CSV_PATH):
        print(f"Error: Train CSV not found at {TRAIN_CSV_PATH}. It is needed to fit the StandardScaler.")
        return

    print("Loading Datasets...")
    
    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_df = pd.read_csv(TRAIN_CSV_PATH)
    train_dataset = CustomMultiModalDataset(train_df, IMG_ROOT, transform=transform, is_train=True)
    
    test_df = pd.read_csv(TEST_CSV_PATH)
    test_dataset = CustomMultiModalDataset(test_df, IMG_ROOT, scaler=train_dataset.scaler, transform=transform, is_train=False)
    test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)
    
    print(f"Test Data: {len(test_df)} samples")
    print("-" * 30)

    model = FusionModel(num_classes=3, fusion_strategy='AFF').to(DEVICE)
    model.load_state_dict(torch.load(MODEL_WEIGHT_PATH, map_location=DEVICE))
    model.eval()

    all_preds = []
    all_labels = []

    print(f"Starting Evaluation on {DEVICE}...")

    with torch.no_grad():
        for images, hc_feats, labels in test_loader:
            images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(images, hc_feats)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    print(f"\n=== Test Results ===")
    print(f"Overall Accuracy: {accuracy:.4f}\n")
    
    print("Classification Report:")
    print(classification_report(all_labels, all_preds, digits=4))

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix (Test Data)')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    
    os.makedirs(SAVE_DIR, exist_ok=True)
    cm_save_path = f'{SAVE_DIR}/test_confusion_matrix.png'
    plt.savefig(cm_save_path)
    plt.close()
    print(f"Confusion Matrix saved to {cm_save_path}")

    save_attention_heatmaps(model, test_loader, DEVICE, SAVE_DIR, num_samples=10)
    print("All tasks completed.")

# --- 実行 ---
if __name__ == "__main__":
    evaluate_model()

Loading Datasets...
Test Data: 144 samples
------------------------------


RuntimeError: Error(s) in loading state_dict for FusionModel:
	Unexpected key(s) in state_dict: "dl_norm.weight", "dl_norm.bias", "dl_norm.running_mean", "dl_norm.running_var", "dl_norm.num_batches_tracked", "hc_projector.1.weight", "hc_projector.1.bias", "hc_projector.1.running_mean", "hc_projector.1.running_var", "hc_projector.1.num_batches_tracked", "classifier.3.weight", "classifier.3.bias", "classifier.2.running_mean", "classifier.2.running_var", "classifier.2.num_batches_tracked". 
	size mismatch for classifier.2.weight: copying a param with shape torch.Size([768]) from checkpoint, the shape in current model is torch.Size([3, 768]).
	size mismatch for classifier.2.bias: copying a param with shape torch.Size([768]) from checkpoint, the shape in current model is torch.Size([3]).

新しいCBAM,Fusionmodelの構造

In [ ]:
import os
import torch
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# --- 1. 設定: ファイルパスとパラメータ ---
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/26_0413_clean'

TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_upsamp/train_max_upsampled.csv'
TEST_CSV_PATH  = f'{pre}/src/FF/AFF/dataset_upsamp/test_fixed.csv'

TARGET_EPOCH = 100
SAVE_DIR = f"../save/CBAMtest/ConvNext_{TARGET_EPOCH}epoch_fixed_dataset"
MODEL_WEIGHT_PATH = f'{SAVE_DIR}/best_fusion_model.pth'


# --- 2. モデルのクラス定義 (学習時と完全に一致させる) ---
class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, initial_weight=0.5):
        super(CBAM, self).__init__()
        reduced_channels = max(1, channels // reduction)
        
        self.fc = nn.Sequential(
            nn.Linear(channels, reduced_channels),
            nn.ReLU(inplace=True),
            nn.Linear(reduced_channels, channels)
        )
        self.sigmoid_channel = nn.Sigmoid()
        
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid_spatial = nn.Sigmoid()

    def forward(self, x):
        # 1. Channel Attention
        avg_out = self.fc(torch.mean(x, dim=(2, 3)))
        max_out = self.fc(torch.amax(x, dim=(2, 3)))
        ch_att = self.sigmoid_channel(avg_out + max_out).unsqueeze(-1).unsqueeze(-1)
        
        # 2. Spatial Attention
        avg_out_sp = torch.mean(x, dim=1, keepdim=True)
        max_out_sp = torch.amax(x, dim=1, keepdim=True)
        sp_att = self.sigmoid_spatial(self.conv_spatial(torch.cat([avg_out_sp, max_out_sp], dim=1)))
        
        attention_weight = ch_att * sp_att
        return attention_weight


class FusionModel(nn.Module):
    def __init__(self, num_classes, fusion_strategy='AFF'):
        super(FusionModel, self).__init__()
        
        convnext = models.convnext_tiny(weights=None)
        self.dl_extractor = convnext.features 
        dl_out_channels = 768
        
        # 学習時に追加した BatchNorm を反映
        self.hc_projector = nn.Sequential(
            nn.Linear(10, dl_out_channels),
            nn.BatchNorm1d(dl_out_channels),
            nn.ReLU(inplace=True)
        )
        
        self.dl_norm = nn.BatchNorm2d(dl_out_channels)
        
        self.fusion_attention = CBAM(dl_out_channels)
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.BatchNorm1d(dl_out_channels),
            nn.Linear(dl_out_channels, num_classes)
        )

    def forward(self, img, hc_vec, return_attention=False):
        # 画像特徴抽出と正規化
        Y = self.dl_extractor(img)
        Y = self.dl_norm(Y)
        
        B, C, H, W = Y.shape
        
        # 数値特徴プロジェクション
        X = self.hc_projector(hc_vec).view(B, C, 1, 1).expand(B, C, H, W)
        
        # 重み計算
        weights = self.fusion_attention(X + Y)
        
        # 融合
        fused = weights * X + (1 - weights) * Y
        out = self.classifier(fused)
        
        if return_attention:
            return out, weights
        return out


# --- 3. Datasetの定義 ---
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler=None, transform=None, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        
        features = self.df[self.feature_cols].values
        
        if is_train:
            self.scaler = StandardScaler()
            self.hc_features = self.scaler.fit_transform(features).astype('float32')
        else:
            assert scaler is not None, "Test set requires a fitted scaler from training set."
            self.scaler = scaler
            self.hc_features = self.scaler.transform(features).astype('float32')
            
        self.labels = self.df['Label'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_root, 'mask', row['category'], row['filename'])
        
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224), (0, 0, 0))
            
        if self.transform:
            image = self.transform(image)
            
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, hc_feat, label


# --- 4. 可視化関数 ---
def save_attention_heatmaps(model, dataloader, device, save_dir, num_samples=10):
    model.eval()
    heatmap_dir = os.path.join(save_dir, 'heatmaps_raw')
    os.makedirs(heatmap_dir, exist_ok=True)

    saved_count = 0
    print(f"\nGenerating {num_samples} raw 7x7 heatmaps to {heatmap_dir}...")

    with torch.no_grad():
        for images, hc_feats, labels in dataloader:
            images, hc_feats = images.to(device), hc_feats.to(device)
            
            outputs, attention_weights = model(images, hc_feats, return_attention=True)
            _, preds = torch.max(outputs, 1)

            for i in range(images.size(0)):
                if saved_count >= num_samples: return

                # チャネル方向に平均して空間マップを取得 (H, W)
                att_map = attention_weights[i].mean(dim=0).cpu().numpy()

                plt.figure(figsize=(6, 5))
                im = plt.imshow(att_map, cmap='binary', vmin=0, vmax=1)
                plt.title(f"Attention Weight (Ratio of Hand-Crafted)\nTrue: {labels[i].item()} | Pred: {preds[i].item()}")
                
                cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
                cbar.set_label('Black(0): CNN / White(1): Hand-Crafted', rotation=270, labelpad=15)
                
                plt.axis('off')
                plt.tight_layout()
                plt.savefig(os.path.join(heatmap_dir, f'heatmap_raw_sample_{saved_count+1}.png'))
                plt.close()
                saved_count += 1


# --- 5. メインテスト関数 ---
def evaluate_model():
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 学習時と同じ224に変更 (256だとズレが生じる可能性があるため)
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    print("Loading Datasets...")
    train_df = pd.read_csv(TRAIN_CSV_PATH)
    train_dataset = CustomMultiModalDataset(train_df, IMG_ROOT, transform=transform, is_train=True)
    
    test_df = pd.read_csv(TEST_CSV_PATH)
    test_dataset = CustomMultiModalDataset(test_df, IMG_ROOT, scaler=train_dataset.scaler, transform=transform, is_train=False)
    test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)
    
    print(f"Test Data: {len(test_df)} samples")
    print("-" * 30)

    # モデル構築と重みロード
    model = FusionModel(num_classes=3).to(DEVICE)
    
    # 厳密にロード (構造が合っていれば通ります)
    state_dict = torch.load(MODEL_WEIGHT_PATH, map_location=DEVICE)
    model.load_state_dict(state_dict)
    model.eval()

    all_preds, all_labels = [], []

    print(f"Starting Evaluation on {DEVICE}...")

    with torch.no_grad():
        for images, hc_feats, labels in test_loader:
            images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
            outputs = model(images, hc_feats)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # 結果表示
    accuracy = accuracy_score(all_labels, all_preds)
    print(f"\n=== Test Results ===")
    print(f"Overall Accuracy: {accuracy:.4f}\n")
    print("Classification Report:")
    print(classification_report(all_labels, all_preds, digits=4))

    # 混同行列
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.savefig(f'{SAVE_DIR}/test_confusion_matrix.png')
    plt.close()

    # アテンションマップ保存
    save_attention_heatmaps(model, test_loader, DEVICE, SAVE_DIR, num_samples=10)
    print("Done.")

if __name__ == "__main__":
    evaluate_model()

Loading Datasets...
Test Data: 144 samples
------------------------------


FileNotFoundError: [Errno 2] No such file or directory: '../save/Concattest/ConvNext_100epoch_fixed_dataset/best_fusion_model.pth'